In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier

df = pd.read_csv("../data/banksim_clean.csv")
df["category"] = df["category"].replace({"Otherservices": "Other Services",
                                         "Hotelservices": "Hotel Services"})
df.shape

(594643, 13)

In [2]:
FEATURES = ["amount", "category", "age_group", "gender"]

pipe = Pipeline([
    ("prep", ColumnTransformer([
        ("num", "passthrough", ["amount"]),
        ("cat", OneHotEncoder(handle_unknown="ignore"), ["category", "age_group", "gender"]),
    ])),
    ("model", RandomForestClassifier(
        n_estimators=100, max_depth=12, min_samples_leaf=5,
        class_weight="balanced_subsample", n_jobs=-1, random_state=42)),
])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
df["risk_score"] = cross_val_predict(pipe, df[FEATURES], df["fraud"],
                                     cv=cv, method="predict_proba")[:, 1]
df["risk_score"].describe()

count    594643.000000
mean          0.071076
std           0.195634
min           0.000921
25%           0.003815
50%           0.004921
75%           0.006541
max           0.998157
Name: risk_score, dtype: float64

In [3]:
THRESHOLD = 0.9

df["risk_level"] = pd.cut(df["risk_score"], bins=[-0.001, 0.5, THRESHOLD, 1.0],
                          labels=["Low", "Medium", "High"])
df["flagged"] = (df["risk_score"] >= THRESHOLD).astype(int)

df["outcome"] = np.select(
    [(df["flagged"] == 1) & (df["fraud"] == 1),
     (df["flagged"] == 1) & (df["fraud"] == 0),
     (df["flagged"] == 0) & (df["fraud"] == 1)],
    ["Caught fraud", "False alarm", "Missed fraud"],
    default="Correctly normal")

# BankSim has no real dates, so we create a SIMULATED date for Power BI's time charts
df["date"] = pd.Timestamp("2022-01-01") + pd.to_timedelta(df["step"], unit="D")

print(df["outcome"].value_counts())
print(df.groupby("risk_level", observed=True)["fraud"].agg(["size", "sum", "mean"]))

outcome
Correctly normal    585109
Caught fraud          5325
False alarm           2334
Missed fraud          1875
Name: count, dtype: int64
              size   sum      mean
risk_level                        
Low         554247   206  0.000372
Medium       32737  1669  0.050982
High          7659  5325  0.695260


In [4]:
import os

cols = ["date", "step", "week", "customer", "merchant", "category", "amount",
        "age_group", "gender", "fraud", "fraud_label",
        "risk_score", "risk_level", "flagged", "outcome"]

df[cols].to_csv("../data/powerbi_data.csv", index=False)
print("Saved:", df[cols].shape)
print("Size (MB):", round(os.path.getsize("../data/powerbi_data.csv") / 1e6, 1))

Saved: (594643, 15)
Size (MB): 75.0
